# load data

In [ ]:
library(broom)
library(scales)
library(stringr)
library(arrow)
library(survival)
library(tidyr)
library(dplyr)
library(haven)
library(reshape2)
library(tidyverse)
library(data.table)
library(lubridate)
library(ggplot2)

In [ ]:
score_kz <- fread("sscore1206_All_combined.csv")

In [ ]:
x <- load("UKB_data_all_1024.Rdata")
x

In [ ]:
data_all_pheno_cox2 <- data_all_pheno_cox2 %>%
  select(-c("Kienb\xf6ck's disease of adults*", "time_Kienb\xf6ck's disease of adults*"))

In [ ]:
data_all_pheno_cox2 <- data_all_pheno_cox2 %>%
  mutate_at(vars(50:2444), ~ replace_na(., 0))

In [ ]:
head(data_all_pheno_cox2)

In [ ]:
fwrite(data_all_pheno_cox2, "data_all_pheno_coxUKB.tsv", sep = "\t", quote = FALSE)

In [ ]:
system("gsutil cp data_all_pheno_coxUKB.tsv gs://bicklab-main-storage/Users/Kun_Zhao")

In [ ]:
data_all_pheno_cox2 <- fread("data_all_pheno_coxUKB.tsv")

In [ ]:
summary(data_all_pheno_cox2$`time_Chronic lymphoid leukemia`)
summary(data_all_pheno_cox2$`time_Acquired deformities of limbs`)

In [ ]:
batch_logistic_regression <- function(data, snp_list, outcome, covariates = NULL) {
  # 初始化结果列表
  results <- list()
  
  # 基础公式部分
  base_formula <- paste0(outcome, " ~ ")
  
  # 循环每个 SNP
  for (snp in snp_list) {
    # 构建公式
    if (!is.null(covariates) && length(covariates) > 0) {
      formula <- as.formula(paste0(base_formula, snp, " + ", paste(covariates, collapse = " + ")))
    } else {
      formula <- as.formula(paste0(base_formula, snp))
    }
    
    # 打印公式以调试
    print(formula)
    
    # 尝试运行模型
    tryCatch({
      logistic_model <- glm(formula, data = data, family = binomial)
      summary_model <- summary(logistic_model)
      
      # 提取模型结果
      coef <- summary_model$coefficients[snp, "Estimate"]
      se <- summary_model$coefficients[snp, "Std. Error"]
      p_value <- summary_model$coefficients[snp, "Pr(>|z|)"]
      ci_lower <- exp(coef - 1.96 * se)
      ci_upper <- exp(coef + 1.96 * se)
      or <- exp(coef)
      
      # 保存结果
      results[[snp]] <- data.frame(
        SNP = snp,
        Beta = coef,
        OR = or,
        SE = se,
        CI_Lower = ci_lower,
        CI_Upper = ci_upper,
        P_Value = p_value
      )
    }, error = function(e) {
      warning(paste("Error in model for SNP:", snp, "- Skipping."))
    })
  }
  
  # 合并所有结果为一个数据框
  results_df <- do.call(rbind, results)
  return(results_df)
}

In [ ]:
pro_list <- colnames(score_kz)[-1]

In [ ]:
summary(score_kz$Interleukin18_receptor_1)
summary(score_kz$Interleukin18binding_protein)

# mCA & CLL

In [ ]:
data_mca <- filter(data_all_pheno_cox2, data_all_pheno_cox2$mca_status==1)

In [ ]:
data_mca <- data_mca[,c("ID_VUMC","cll","surv_cll","baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")]

In [ ]:
data_mca_protein <- merge(data_mca, score_kz, by.x = "ID_VUMC" , by.y = "ID", all.x = T)

In [ ]:
head(data_mca_protein)

In [ ]:
data_mca_protein <- filter(data_mca_protein, data_mca_protein$surv_cll > 0)

In [ ]:
dim(data_mca_protein)

In [ ]:
table(data_mca_protein$cll, useNA= "always")

In [ ]:
#association

In [ ]:
length(pro_list)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_mca_protein,
  snp_list = pro_list,
  outcome = "cll",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
logistic_mcasig <- filter(logistic_mca, logistic_mca$P_Value <= 0.05)
logistic_mcasig

In [ ]:
logistic_mca2 <- filter(logistic_mca, logistic_mca$SNP == "Interleukin6")
logistic_mca2

In [ ]:
logistic_mca2 <- filter(logistic_mca, logistic_mca$SNP == "Tumor_necrosis_factor_receptor_superfamily_member_1A")
logistic_mca2

# Overall CHIP

In [ ]:
calls_chip <- fread('All_CHIP_calls_for_UKB_March112024.txt') #17026

In [ ]:
x <- load("Users_Kun_Zhao_vumc_id_ukb.Rdata")

In [ ]:
calls_chip <- merge(calls_chip,vumc_id, by.x = "ID", by.y = "ID_Broad", all.x = T)

In [ ]:
table(data_all_pheno_cox2$chip)
table(data_all_pheno_cox2$`Acute myeloid leukemia`)
table(data_all_pheno_cox2$`Coronary atherosclerosis [Atherosclerotic heart disease]`)
table(data_all_pheno_cox2$`Myocardial infarction [Heart attack]`)
table(data_all_pheno_cox2$`Myelodysplastic syndrome`)

In [ ]:
data_chip <- filter(data_all_pheno_cox2, data_all_pheno_cox2$chip==1)

In [ ]:
table(data_chip$`Myeloproliferative disorder`)

In [ ]:
data_chip$CVD <- NA
data_chip$CVD <- ifelse(data_chip$`Coronary atherosclerosis [Atherosclerotic heart disease]`==1|
                        data_chip$`Myocardial infarction [Heart attack]`==1 |
                        data_chip$Stroke==1|
                        data_chip$`Heart failure`==1|
                        data_chip$`Angina pectoris`==1|
                        data_chip$`Peripheral vascular disease`==1,1, 0)

In [ ]:
table(data_chip$CVD)

In [ ]:
data_chip$ASCVD <- NA
data_chip$ASCVD <- ifelse(data_chip$`Coronary atherosclerosis [Atherosclerotic heart disease]`==1|
                        data_chip$`Myocardial infarction [Heart attack]`==1 |
                        data_chip$`Heart failure`==1|
                        data_chip$`Angina pectoris`==1|
                        data_chip$`Peripheral vascular disease`==1,1, 0)

In [ ]:
data_chip$CAD <- NA
data_chip$CAD <- ifelse(data_chip$`Coronary atherosclerosis [Atherosclerotic heart disease]`==1|
                        data_chip$`Myocardial infarction [Heart attack]`==1 |
                        data_chip$`Angina pectoris`==1,1, 0)

In [ ]:
data_chip$AML <- NA
data_chip$AML <- ifelse(data_chip$`Acute myeloid leukemia`==1|
                        data_chip$`Myelodysplastic syndrome`==1 |
                        data_chip$Myelofibrosis==1|
                        data_chip$`Myeloproliferative disorder`==1|
                        data_chip$`Polycythemia vera`==1|
                        data_chip$`Essential thrombocythemia`==1,1, 0)

In [ ]:
data_chip$AID <- NA
data_chip$AID <- ifelse(data_chip$`Rheumatoid arthritis`==1|
                        data_chip$`Ankylosing spondylitis`==1 |
                        data_chip$`Systemic lupus erythematosus [SLE]`==1|
                        data_chip$Psoriasis==1|
                        data_chip$`Psoriatic arthropathy`==1,1, 0)
table(data_chip$AID)

In [ ]:
data_chip$VTE <- NA
data_chip$VTE <- ifelse(data_chip$`Pulmonary embolism`==1|
                        data_chip$`Venous thromboembolism`==1 ,1, 0)

In [ ]:
data_chip$MPN <- NA
data_chip$MPN <- ifelse(data_chip$`Myeloproliferative disorder`==1|
                        data_chip$VTE==1 |
                        data_chip$`Chronic myeloproliferative disease*` == 1 ,1, 0)

In [ ]:
data_chip2 <- data_chip[,c("ID_VUMC","baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_HTN","baseline_DM","systolicBP_0","diastolicBP_0","ldl_0",
                          "Acute myeloid leukemia","Heart failure","AML","ASCVD","CVD","CAD","Myocardial infarction [Heart attack]","Stroke", "Thrombosis of cerebral or precerebral arteries","Venous thromboembolism","VTE","death","AID",
                          "Rheumatoid arthritis","Ankylosing spondylitis","Systemic lupus erythematosus [SLE]","Psoriasis","Psoriatic arthropathy","Myeloproliferative disorder", "MPN","Chronic myeloproliferative disease*")]

In [ ]:
score_kz$ID <- as.character(score_kz$ID)
data_chip2$ID_VUMC <- as.character(data_chip2$ID_VUMC)

In [ ]:
data_chip_protein <- merge(data_chip2, score_kz, by.x = "ID_VUMC" , by.y = "ID", all.x = T)

In [ ]:
dim(data_chip_protein)

In [ ]:
head(data_chip_protein)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_protein,
  snp_list = pro_list,
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","systolicBP_0","diastolicBP_0","ldl_0")
)

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
il17r <- filter(logistic_mca, logistic_mca$SNP == "Interleukin17_receptor_B" | logistic_mca$SNP == "Interleukin17_receptor_A")
il17r

In [ ]:
write.csv(logistic_mca, file="AllCHIP_CVD_protein_adjusted_UKB.csv")

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_protein,
  snp_list = "Interleukin18binding_protein",
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","systolicBP_0","diastolicBP_0","ldl_0")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_protein,
  snp_list = pro_list,
  outcome = "ASCVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
write.csv(logistic_mca, file="CHIP_ASCVD_protein.csv")

In [ ]:
table(data_chip_protein$`Heart failure`)

In [ ]:
logistic_mca <- NA

logistic_mca <- batch_logistic_regression(
  data = data_chip_protein,
  snp_list = pro_list,
  outcome = "`Heart failure`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
write.csv(logistic_mca, file="CHIP_HF_protein.csv")

In [ ]:
logistic_mca <- NA

logistic_mca <- batch_logistic_regression(
  data = data_chip_protein,
  snp_list = pro_list,
  outcome = "CAD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
write.csv(logistic_mca, file="CHIP_CAD_protein.csv")

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_protein,
  snp_list = pro_list,
  outcome = "`Acute myeloid leukemia`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_protein,
  snp_list = pro_list,
  outcome = "AML",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

# TET2

In [ ]:
d3a <- filter(calls_chip, calls_chip$Gene.refGene == "DNMT3A")
tet2 <-filter(calls_chip, calls_chip$Gene.refGene == "TET2")
jak2 <-filter(calls_chip, calls_chip$Gene.refGene == "JAK2")

In [ ]:
data_chip_protein$d3a <- ifelse(data_chip_protein$ID_VUMC %in% d3a$ID_VUMC , 1, 0)
data_chip_protein$tet2 <- ifelse(data_chip_protein$ID_VUMC %in% tet2$ID_VUMC , 1, 0)
data_chip_protein$jak2 <- ifelse(data_chip_protein$ID_VUMC %in% jak2$ID_VUMC , 1, 0)

In [ ]:
logistic_jak2 <- NA
logistic_jak2 <- batch_logistic_regression(
  data = data_chip_protein,
  snp_list = "Interleukin17_receptor_A",
  outcome = "jak2",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_jak2

In [ ]:
summary(data_chip_protein$Interleukin17_receptor_A)

In [ ]:
data_chip_protein2 <- filter(data_chip_protein,is.na(data_chip_protein$Interleukin17_receptor_A)==F)
data_chip_protein2$il17ra_low <- 0
data_chip_protein2$il17ra_low[data_chip_protein2$Interleukin17_receptor_A < quantile(data_chip_protein2$Interleukin17_receptor_A,0.2)] <- 1
table(data_chip_protein2$il17ra_low)

In [ ]:
table(data_chip_protein2$il17ra_low, data_chip_protein2$jak2)

In [ ]:
logistic_jak2 <- NA
logistic_jak2 <- batch_logistic_regression(
  data = data_chip_protein2,
  snp_list = "il17ra_low",
  outcome = "jak2",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_jak2

In [ ]:
data_tet2 <- filter(data_chip_protein, data_chip_protein$tet2 == 1)

In [ ]:
logistic_mca <- NA

logistic_mca <- batch_logistic_regression(
  data = data_tet2,
  snp_list = pro_list,
  outcome = "CAD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
write.csv(logistic_mca,file="tet2_cad_omicspred_ukb.csv")

In [ ]:
logistic_mca <- NA

logistic_mca <- batch_logistic_regression(
  data = data_tet2,
  snp_list = pro_list,
  outcome = "ASCVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
logistic_mca <- NA

logistic_mca <- batch_logistic_regression(
  data = data_tet2,
  snp_list = pro_list,
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

# JAK2

In [ ]:
data_jak2 <- filter(data_chip_protein, data_chip_protein$jak2 == 1)

In [ ]:
table(data_jak2$CVD, useNA= "always")
table(data_jak2$AID, useNA= "always")
table(data_jak2$`Myeloproliferative disorder`, useNA= "always")

In [ ]:
data_jak2$CVDandDeath <- NA
data_jak2$CVDandDeath <- ifelse(data_jak2$CVD == 1 | data_jak2$death == 1, 1, 0)
table(data_jak2$CVDandDeath)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = pro_list,
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = pro_list,
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","baseline_HTN","ldl_0")
)

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
il17r <- filter(logistic_mca, logistic_mca$SNP == "Interleukin17_receptor_B" | logistic_mca$SNP == "Interleukin17_receptor_A")
il17r

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = "Interleukin17_receptor_A",
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","baseline_HTN","ldl_0","AID")
)
logistic_mca

In [ ]:
table(data_jak2$Interleukin18binding_protein)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = "Interleukin18binding_protein",
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","baseline_HTN","ldl_0")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = "Interleukin18_receptor_1",
  outcome = "VTE",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","baseline_HTN","ldl_0")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = "Interleukin17_receptor_A",
  outcome = "AID",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","baseline_HTN","ldl_0")
)
logistic_mca

In [ ]:
table(data_all_pheno_cox2$`Myeloproliferative disorder`)
table(data_jak2$`Myeloproliferative disorder`)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = c("Interleukin17_receptor_A","Interleukin17_receptor_B"),
  outcome = "`Myeloproliferative disorder`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","baseline_HTN","ldl_0","AID")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = c("Interleukin17_receptor_A","Interleukin17_receptor_B"),
  outcome = "MPN",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","baseline_HTN","ldl_0","AID")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = c("Interleukin17_receptor_A","Interleukin17_receptor_B"),
  outcome = "CVDandDeath",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","baseline_HTN","ldl_0","AID")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = pro_list,
  outcome = "Stroke",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = pro_list,
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = pro_list,
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","systolicBP_0","diastolicBP_0","baseline_DM")
)

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
test <- filter(logistic_mca, logistic_mca$SNP == "Interleukin17_receptor_A")
test

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = pro_list,
  outcome = "VTE",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

# PheWAS IL17RA in JAK2

In [ ]:
disease_cols <- colnames(data_chip)[50:2444]
disease_cols

In [ ]:
data_jak2_allpheno <- filter(data_chip, data_chip$

# VTE & CVD

#All CHIP

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_protein,
  snp_list = c("Interleukin17_receptor_B","Interleukin17_receptor_A"),
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","systolicBP_0","diastolicBP_0","baseline_DM")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_protein,
  snp_list = c("Interleukin17_receptor_B","Interleukin17_receptor_A"),
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","systolicBP_0","diastolicBP_0","baseline_DM","AID")
)
logistic_mca

#Other CHIP

In [ ]:
data_chip_nonjak2 <-  filter(data_chip_protein, data_chip_protein$jak2 == 0)

In [ ]:
table(data_chip_nonjak2$CVD, useNA= "always")

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_nonjak2,
  snp_list = c("Interleukin17_receptor_B","Interleukin17_receptor_A"),
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","systolicBP_0","diastolicBP_0","baseline_DM")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_nonjak2,
  snp_list = pro_list,
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","baseline_HTN","ldl_0")
)
logistic_mca

In [ ]:
il17r <- filter(logistic_mca, logistic_mca$SNP == "Interleukin17_receptor_B" | logistic_mca$SNP == "Interleukin17_receptor_A")
il17r

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_nonjak2,
  snp_list = "Interleukin17_receptor_A",
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","baseline_HTN","ldl_0","AID")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_nonjak2,
  snp_list = c("Interleukin18_receptor_1","Interleukin18binding_protein"),
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","systolicBP_0","diastolicBP_0","baseline_DM")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_nonjak2,
  snp_list = c("Interleukin18_receptor_1","Interleukin18binding_protein"),
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","baseline_HTN","ldl_0")
)
logistic_mca

#All People

In [ ]:
data_all_pheno_cox2$CVD <- NA
data_all_pheno_cox2$CVD <- ifelse(data_all_pheno_cox2$`Coronary atherosclerosis [Atherosclerotic heart disease]`==1|
                        data_all_pheno_cox2$`Myocardial infarction [Heart attack]`==1 |
                        data_all_pheno_cox2$Stroke==1|
                        data_all_pheno_cox2$`Heart failure`==1|
                        data_all_pheno_cox2$`Angina pectoris`==1|
                        data_all_pheno_cox2$`Peripheral vascular disease`==1,1, 0)
data_all_pheno_cox2$CVD <- ifelse(is.na(data_all_pheno_cox2$CVD),0,data_all_pheno_cox2$CVD)

In [ ]:
data_all_pheno_cox2$AID <- NA
data_all_pheno_cox2$AID <- ifelse(data_all_pheno_cox2$`Rheumatoid arthritis`==1|
                        data_all_pheno_cox2$`Ankylosing spondylitis`==1 |
                        data_all_pheno_cox2$`Systemic lupus erythematosus [SLE]`==1|
                        data_all_pheno_cox2$Psoriasis==1|
                        data_all_pheno_cox2$`Psoriatic arthropathy`==1,1, 0)
data_all_pheno_cox2$AID <- ifelse(is.na(data_all_pheno_cox2$AID),0,data_all_pheno_cox2$AID)
table(data_all_pheno_cox2$AID, useNA = "always")

In [ ]:
data_VTE <- data_all_pheno_cox2[,c("ID_VUMC","chip","baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5",
                                   "Venous thromboembolism","CVD","death","BMI_0","baseline_HTN","baseline_DM","systolicBP_0","diastolicBP_0","ldl_0","AID",
                                  "Rheumatoid arthritis","Ankylosing spondylitis","Systemic lupus erythematosus [SLE]","Psoriasis","Psoriatic arthropathy")]

In [ ]:
table(data_VTE$`Venous thromboembolism`,useNA="always")
table(data_VTE$CVD,useNA="always")
table(data_VTE$Psoriasis,useNA="always")

In [ ]:
data_VTE$ID_VUMC <- as.character(data_VTE$ID_VUMC)

In [ ]:
data_VTE_protein <- merge(data_VTE, score_kz, by.x = "ID_VUMC" , by.y = "ID", all.x = T)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_VTE_protein,
  snp_list = c("Interleukin17_receptor_B","Interleukin17_receptor_A"),
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","baseline_HTN","baseline_DM")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_VTE_protein,
  snp_list = c("Interleukin17_receptor_B","Interleukin17_receptor_A"),
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","baseline_HTN","baseline_DM")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_VTE_protein,
  snp_list = c("Interleukin17_receptor_B","Interleukin17_receptor_A"),
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","systolicBP_0","diastolicBP_0","ldl_0","AID")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_VTE_protein,
  snp_list = c("Interleukin18_receptor_1","Interleukin18binding_protein"),
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","systolicBP_0","diastolicBP_0","ldl_0")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_VTE_protein,
  snp_list = c("Interleukin17_receptor_B","Interleukin17_receptor_A"),
  outcome = "Psoriasis",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","systolicBP_0","diastolicBP_0","ldl_0")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_VTE_protein,
  snp_list = "Interleukin17_receptor_B",
  outcome = "death",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

#Non-CHIP

In [ ]:
data_VTE_nonchip <- filter(data_VTE_protein, data_VTE_protein$chip == 0)

In [ ]:
dim(data_VTE_nonchip)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_VTE_nonchip,
  snp_list = c("Interleukin17_receptor_B","Interleukin17_receptor_A"),
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","baseline_HTN","baseline_DM")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_VTE_nonchip,
  snp_list = c("Interleukin17_receptor_B","Interleukin17_receptor_A"),
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","BMI_0","baseline_DM","systolicBP_0","diastolicBP_0","ldl_0","AID")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_VTE_nonchip,
  snp_list = "Interleukin17_receptor_B",
  outcome = "death",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

# IL17R measurement

In [ ]:
score_yp <- fread("ukb_52k_proteome_dec12_yp.tsv")

In [ ]:
summary(score_yp$il18)
summary(score_yp$il18bp)
summary(score_yp$il18r1)
summary(score_yp$il18rap)

In [ ]:
il17r <- score_yp[,c("eid","il17ra","il17rb")]

In [ ]:
il18 <- score_yp[,c("eid","il18","il18bp","il18r1","il18rap")]

In [ ]:
#All people

In [ ]:
data_VTE2 <- merge(data_VTE, il17r, by.x = "ID_VUMC", by.y = "eid", all = F)

In [ ]:
il18$eid <- as.character(il18$eid)

In [ ]:
dim(il18)

In [ ]:
data_il18_meas<- merge(data_VTE, il18, by.x = "ID_VUMC", by.y = "eid", all = F)
dim(data_il18_meas)

In [ ]:
dim(data_VTE2)

In [ ]:
summary(data_VTE2$il17ra)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_il18_meas,
  snp_list = c("il18r1","il18","il18bp","il18rap"),
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","baseline_HTN","baseline_DM")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_il18_meas,
  snp_list = "il17ra",
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
#jak2

In [ ]:
data_jak2_2 <- merge(data_jak2, il17r, by.x = "ID_VUMC", by.y = "eid", all = F)
dim(data_jak2_2)

In [ ]:
data_jak2_il18 <- merge(data_jak2, il18, by.x = "ID_VUMC", by.y = "eid", all = F)
dim(data_jak2_il18)

In [ ]:
table(data_jak2_il18$`Venous thromboembolism`)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2_il18,
  snp_list = c("il18r1","il18","il18bp","il18rap"),
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","baseline_HTN","baseline_DM")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2_2,
  snp_list = "il17rb",
  outcome = "VTE",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
#all

In [ ]:
data_all_pheno_cox3 <- merge(data_all_pheno_cox2, il17r, by.x = "ID_VUMC", by.y = "eid", all = F)
dim(data_all_pheno_cox3)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_all_pheno_cox3,
  snp_list = "il17rb",
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
#CHIP

In [ ]:
data_chip$ID_VUMC <- as.character(data_chip$ID_VUMC)

In [ ]:
data_chip_il18 <- merge(data_chip, il18, by.x = "ID_VUMC", by.y = "eid", all = F)
dim(data_chip_il18)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chip_il18,
  snp_list = c("il18r1","il18","il18bp","il18rap"),
  outcome = "VTE",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","baseline_HTN","baseline_DM")
)
logistic_mca

# CHIP mCA & VTE (HR)

In [ ]:
dim(data_all_pheno_cox2)

In [ ]:
table(data_all_pheno_cox2$`Venous thromboembolism`,useNA = "always")
summary(data_all_pheno_cox2$`time_Venous thromboembolism`)
summary(data_all_pheno_cox2$min_date)
summary(data_all_pheno_cox2$max_date)

In [ ]:
data_all_pheno_cox2$surv_VTE <- NA
data_all_pheno_cox2$surv_all <- NA
data_all_pheno_cox2$surv_VTE <- as.numeric(difftime(data_all_pheno_cox2$`time_Venous thromboembolism`, data_all_pheno_cox2$min_date, units ="days"))/365.25
data_all_pheno_cox2$surv_all <- as.numeric(difftime(data_all_pheno_cox2$max_date, data_all_pheno_cox2$min_date, units ="days"))/365.25
summary(data_all_pheno_cox2$surv_VTE)
data_all_pheno_cox2$surv_VTE <- ifelse(is.na(data_all_pheno_cox2$surv_VTE),data_all_pheno_cox2$surv_all,data_all_pheno_cox2$surv_VTE)
summary(data_all_pheno_cox2$surv_VTE)

In [ ]:
data_all_pheno_cox2$jak2 <- ifelse(data_all_pheno_cox2$ID_VUMC %in% jak2$ID_VUMC , 1, 0)
table(data_all_pheno_cox2$jak2)

In [ ]:
data_all_pheno_cox2$`Venous thromboembolism` <- as.factor(data_all_pheno_cox2$`Venous thromboembolism`)

In [ ]:
data_all_pheno_cox2$VTE <- NA
data_all_pheno_cox2$VTE <- ifelse(data_all_pheno_cox2$`Pulmonary embolism`==1|
                                  data_all_pheno_cox2$`Venous thromboembolism`==1 ,1, 0)

In [ ]:
data_all_pheno_cox2$surv_PE <- NA
data_all_pheno_cox2$surv_PE <- as.numeric(difftime(data_all_pheno_cox2$`time_Pulmonary embolism`, data_all_pheno_cox2$min_date, units ="days"))/365.25
summary(data_all_pheno_cox2$surv_PE)
data_all_pheno_cox2$surv_PE <- ifelse(is.na(data_all_pheno_cox2$surv_PE),data_all_pheno_cox2$surv_all,data_all_pheno_cox2$surv_PE)
summary(data_all_pheno_cox2$surv_PE)

In [ ]:
data_all_pheno_cox2 <- data_all_pheno_cox2 %>% mutate(surv_VTE2 = pmin(surv_PE,surv_VTE))

In [ ]:
summary(data_all_pheno_cox2$surv_VTE2)
summary(data_all_pheno_cox2$surv_VTE)
summary(data_all_pheno_cox2$surv_PE)

In [ ]:
data_all_pheno_cox2$surv_AHD <- NA
data_all_pheno_cox2$surv_AHD <- as.numeric(difftime(data_all_pheno_cox2$`time_Coronary atherosclerosis [Atherosclerotic heart disease]`, data_all_pheno_cox2$min_date, units ="days"))/365.25
summary(data_all_pheno_cox2$surv_AHD)
data_all_pheno_cox2$surv_AHD <- ifelse(is.na(data_all_pheno_cox2$surv_AHD),data_all_pheno_cox2$surv_all,data_all_pheno_cox2$surv_AHD)
summary(data_all_pheno_cox2$surv_AHD)

In [ ]:
data_all_pheno_cox2$surv_AMI <- NA
data_all_pheno_cox2$surv_AMI <- as.numeric(difftime(data_all_pheno_cox2$`time_Myocardial infarction [Heart attack]`, data_all_pheno_cox2$min_date, units ="days"))/365.25
summary(data_all_pheno_cox2$surv_AMI)
data_all_pheno_cox2$surv_AMI <- ifelse(is.na(data_all_pheno_cox2$surv_AMI),data_all_pheno_cox2$surv_all,data_all_pheno_cox2$surv_AMI)
summary(data_all_pheno_cox2$surv_AMI)

In [ ]:
data_all_pheno_cox2$surv_str <- NA
data_all_pheno_cox2$surv_str <- as.numeric(difftime(data_all_pheno_cox2$`time_Stroke`, data_all_pheno_cox2$min_date, units ="days"))/365.25
summary(data_all_pheno_cox2$surv_str)
data_all_pheno_cox2$surv_str <- ifelse(is.na(data_all_pheno_cox2$surv_str),data_all_pheno_cox2$surv_all,data_all_pheno_cox2$surv_str)
summary(data_all_pheno_cox2$surv_str)

In [ ]:
data_all_pheno_cox2$surv_hf <- NA
data_all_pheno_cox2$surv_hf <- as.numeric(difftime(data_all_pheno_cox2$`time_Heart failure`, data_all_pheno_cox2$min_date, units ="days"))/365.25
summary(data_all_pheno_cox2$surv_hf)
data_all_pheno_cox2$surv_hf <- ifelse(is.na(data_all_pheno_cox2$surv_hf),data_all_pheno_cox2$surv_all,data_all_pheno_cox2$surv_hf)
summary(data_all_pheno_cox2$surv_hf)

In [ ]:
data_all_pheno_cox2$surv_AP <- NA
data_all_pheno_cox2$surv_AP <- as.numeric(difftime(data_all_pheno_cox2$`time_Angina pectoris`, data_all_pheno_cox2$min_date, units ="days"))/365.25
summary(data_all_pheno_cox2$surv_AP)
data_all_pheno_cox2$surv_AP <- ifelse(is.na(data_all_pheno_cox2$surv_AP),data_all_pheno_cox2$surv_all,data_all_pheno_cox2$surv_AP)
summary(data_all_pheno_cox2$surv_AP)

In [ ]:
data_all_pheno_cox2$surv_PVD <- NA
data_all_pheno_cox2$surv_PVD <- as.numeric(difftime(data_all_pheno_cox2$`time_Peripheral vascular disease`, data_all_pheno_cox2$min_date, units ="days"))/365.25
summary(data_all_pheno_cox2$surv_PVD)
data_all_pheno_cox2$surv_PVD <- ifelse(is.na(data_all_pheno_cox2$surv_PVD),data_all_pheno_cox2$surv_all,data_all_pheno_cox2$surv_PVD)
summary(data_all_pheno_cox2$surv_PVD)

In [ ]:
data_all_pheno_cox2 <- data_all_pheno_cox2 %>% mutate(surv_CVD = pmin(surv_PVD,surv_AP,surv_hf,surv_str,surv_AMI,surv_AHD))

In [ ]:
summary(data_all_pheno_cox2$surv_CVD)

In [ ]:
data_all_pheno_cox2$VTE <- as.factor(data_all_pheno_cox2$VTE)

In [ ]:
table(data_all_pheno_cox2$VTE,useNA = "always")
class(data_all_pheno_cox2$VTE)

In [ ]:
analyze_cox_model <- function(data, time_var, event_var, group_var, covariates) {
  # 过滤掉 time_var 小于等于 0 的数据
  df_filtered <- data %>% filter(.data[[time_var]] > 0)
    
  # 确保 time_var 和 event_var 长度一致
  if (nrow(df_filtered) != length(df_filtered[[time_var]]) || 
      nrow(df_filtered) != length(df_filtered[[event_var]])) {
    stop("Time and status are different lengths after filtering. Please check your data.")
  }
  
  # 检查 event_var 是否为二值变量
  if (!all(unique(df_filtered[[event_var]]) %in% c(0, 1))) {
    stop("Event variable must be binary (0 or 1).")
  }
  
  # 构建生存分析对象
  surv_obj <- Surv(time = df_filtered[[time_var]], event = df_filtered[[event_var]] == 1)
  
  # 构建公式
  covariate_formula <- paste(covariates, collapse = " + ")
  formula <- as.formula(paste("surv_obj ~", group_var, "+", covariate_formula))
  
  # 运行 Cox 模型
  cox_fit <- coxph(formula, data = df_filtered)
  
  # 提取和打印结果
  summary_cox <- summary(cox_fit)
  print(summary_cox)
  
  # 结果格式化
  results <- as.data.frame(summary_cox$coefficients)
  colnames(results) <- c("Coefficient", "Exp(Coefficient)", "Standard Error", "z value", "p value")
  print(results)
  
  return(results)
}

In [ ]:
data_all_pheno_cox2$jak2 <- ifelse(data_all_pheno_cox2$chip == 1 & data_all_pheno_cox2$jak2 == 0, NA, data_all_pheno_cox2$jak2)

In [ ]:
results <- analyze_cox_model(
  data = data_all_pheno_cox2, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "jak2", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5")
)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_all_pheno_cox2,
  snp_list = "jak2",
  outcome = "VTE",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5")
)
logistic_mca

In [ ]:
table(data_all_pheno_cox2$mca_status)
dim(data_all_pheno_cox2)

In [ ]:
calls_mca <- fread('mocha_ukb_autosomal_mca_calls.tsv') #17865

In [ ]:
calls_mca2 <- calls_mca[,c("ID_VUMC","chrom","type")]

In [ ]:
chr9CN <- filter(calls_mca2, calls_mca2$chrom == "chr9" & calls_mca2$type == "CN-LOH")

In [ ]:
chr9loss <- filter(calls_mca2, calls_mca2$chrom == "chr9" & calls_mca2$type == "Loss")

In [ ]:
chr9 <- filter(calls_mca2, calls_mca2$chrom == "chr9")

In [ ]:
data_all_pheno_cox2$chr9CNLOH <- NA
data_all_pheno_cox2$chr9CNLOH <- ifelse(data_all_pheno_cox2$ID_VUMC %in% chr9CN$ID_VUMC,1,0)

In [ ]:
data_all_pheno_cox2$chr9Loss <- NA
data_all_pheno_cox2$chr9Loss <- ifelse(data_all_pheno_cox2$ID_VUMC %in% chr9loss$ID_VUMC,1,0)

In [ ]:
data_all_pheno_cox2$chr9 <- NA
data_all_pheno_cox2$chr9 <- ifelse(data_all_pheno_cox2$ID_VUMC %in% chr9$ID_VUMC,1,0)

In [ ]:
table(data_all_pheno_cox2$chr9CNLOH, useNA = "always")
table(data_all_pheno_cox2$chr9Loss, useNA = "always")
table(data_all_pheno_cox2$chr9, useNA = "always")

In [ ]:
data_all_pheno_cox2$chr9CNLOH <- ifelse(data_all_pheno_cox2$mca_status == 1 & data_all_pheno_cox2$chr9CNLOH == 0, NA, )

In [ ]:
data_all_pheno_cox2$chr9C_L <- ifelse(data_all_pheno_cox2$chr9CNLOH == 1 | data_all_pheno_cox2$chr9Loss == 1, 1, 0)

In [ ]:
table(data_all_pheno_cox2$chr9CNLOH, useNA = "always")
table(data_all_pheno_cox2$chr9C_L, useNA = "always")

In [ ]:
results <- analyze_cox_model(
  data = data_all_pheno_cox2, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "chr9CNLOH", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5")
)

In [ ]:
data_all_pheno_cox2$chr9CN_only <- NA
data_all_pheno_cox2$chr9CN_only <- ifelse(data_all_pheno_cox2$chr9CNLOH == 1,1,data_all_pheno_cox2$chr9CN_only)
data_all_pheno_cox2$chr9CN_only <- ifelse(data_all_pheno_cox2$jak2 == 0 & data_all_pheno_cox2$chr9CNLOH == 0,0,data_all_pheno_cox2$chr9CN_only)
table(data_all_pheno_cox2$chr9CN_only)

In [ ]:
results <- analyze_cox_model(
  data = data_all_pheno_cox2, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "chr9CN_only", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5")
)

In [ ]:
data_all_pheno_cox2$combine_jak2_chr9 <- NA
data_all_pheno_cox2$combine_jak2_chr9 <- ifelse(data_all_pheno_cox2$jak2 == 1 |  data_all_pheno_cox2$chr9 == 1 ,1, 0)
data_all_pheno_cox2$combine_jak2_chr9 <- ifelse(data_all_pheno_cox2$jak2_only == 0 &  data_all_pheno_cox2$chip == 1 ,0, data_all_pheno_cox2$combine_jak2_chr9)
data_all_pheno_cox2$combine_jak2_chr9 <- ifelse(data_all_pheno_cox2$chr9 == 0 &  data_all_pheno_cox2$mca_status == 1 ,0, data_all_pheno_cox2$combine_jak2_chr9)

In [ ]:
table(data_all_pheno_cox2$combine_jak2_chr9)

In [ ]:
data_all_pheno_cox2$combine_jak2_chr9cn <- NA
data_all_pheno_cox2$combine_jak2_chr9cn <- ifelse(data_all_pheno_cox2$jak2_only == 1 |  data_all_pheno_cox2$chr9CN_only == 1 ,1, 0)
#data_all_pheno_cox2$combine_jak2_chr9cn <- ifelse(data_all_pheno_cox2$jak2_only == 0 ,0, data_all_pheno_cox2$combine_jak2_chr9cn)

In [ ]:
table(data_all_pheno_cox2$combine_jak2_chr9cn,useNA = "always")

In [ ]:
results <- analyze_cox_model(
  data = data_all_pheno_cox2, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "combine_jak2_chr9cn", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5")
)

In [ ]:
data_all_pheno_cox2$jak2_only <- NA
data_all_pheno_cox2$jak2_only <- ifelse(data_all_pheno_cox2$jak2 == 1,1,data_all_pheno_cox2$jak2_only)
data_all_pheno_cox2$jak2_only <- ifelse(data_all_pheno_cox2$jak2 == 0 & data_all_pheno_cox2$chr9CNLOH == 0,0,data_all_pheno_cox2$jak2_only)

In [ ]:
table(data_all_pheno_cox2$jak2_only,useNA = "always")

In [ ]:
results <- analyze_cox_model(
  data = data_all_pheno_cox2, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "jak2_only", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5")
)

In [ ]:
data_all_pheno_cox2$both_jak2_chr9cn <- NA
data_all_pheno_cox2$both_jak2_chr9cn <- ifelse(data_all_pheno_cox2$jak2_only == 1 & data_all_pheno_cox2$chr9CN_only == 1 ,1, 0)

In [ ]:
table(data_all_pheno_cox2$both_jak2_chr9cn, useNA = "always")

In [ ]:
results <- analyze_cox_model(
  data = data_all_pheno_cox2, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "both_jak2_chr9cn", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5")
)

# IL17R & CVDgroup (categories)

In [ ]:
il17rb <- score_kz[,c("ID","Interleukin17_receptor_B","Interleukin17_receptor_A")]

In [ ]:
table(data_all_pheno_cox2$CVD)

In [ ]:
data_all_pheno_cox2$ID_VUMC <- as.character(data_all_pheno_cox2$ID_VUMC)
il17rb$ID <- as.character(il17rb$ID)
data_all_pheno_cox3 <- merge(il17rb, data_all_pheno_cox2, by.x = "ID", by.y = "ID_VUMC" , all.x = T)

In [ ]:
summary(data_all_pheno_cox3$Interleukin17_receptor_B)

In [ ]:
data_all_pheno_cox3$il17rb_262 <- 2
data_all_pheno_cox3$il17rb_262[data_all_pheno_cox3$Interleukin17_receptor_B >= quantile(data_all_pheno_cox3$Interleukin17_receptor_B,0.8)] <- 3
data_all_pheno_cox3$il17rb_262[data_all_pheno_cox3$Interleukin17_receptor_B < quantile(data_all_pheno_cox3$Interleukin17_receptor_B,0.2)] <- 1
data_all_pheno_cox3$il17rb_262<-as.factor(data_all_pheno_cox3$il17rb_262)
table(data_all_pheno_cox3$il17rb_262)

In [ ]:
data_all_pheno_cox3$il17rb_333 <- 2
data_all_pheno_cox3$il17rb_333[data_all_pheno_cox3$Interleukin17_receptor_B >= quantile(data_all_pheno_cox3$Interleukin17_receptor_B,2/3)] <- 3
data_all_pheno_cox3$il17rb_333[data_all_pheno_cox3$Interleukin17_receptor_B < quantile(data_all_pheno_cox3$Interleukin17_receptor_B,1/3)] <- 1
data_all_pheno_cox3$il17rb_333 <- as.factor(data_all_pheno_cox3$il17rb_333)
table(data_all_pheno_cox3$il17rb_333)

In [ ]:
data_all_pheno_cox3$il17ra_262 <- 2
data_all_pheno_cox3$il17ra_262[data_all_pheno_cox3$Interleukin17_receptor_A >= quantile(data_all_pheno_cox3$Interleukin17_receptor_A,0.8)] <- 3
data_all_pheno_cox3$il17ra_262[data_all_pheno_cox3$Interleukin17_receptor_A < quantile(data_all_pheno_cox3$Interleukin17_receptor_A,0.2)] <- 1
data_all_pheno_cox3$il17ra_262<-as.factor(data_all_pheno_cox3$il17r_262)
table(data_all_pheno_cox3$il17ra_262)

In [ ]:
table(data_all_pheno_cox2$CVD,useNA = "always")

In [ ]:
data_il17rb <- data_all_pheno_cox3[,c("ID","il17rb_262","il17ra_262","il17rb_333","Interleukin17_receptor_B","Interleukin17_receptor_A","Venous thromboembolism","baseline_age", "age2", "genetic_sex",
                                     "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","baseline_DM","baseline_HTN","BMI_0","systolicBP_0","diastolicBP_0",
                                     "VTE","surv_VTE","surv_VTE2","chip","jak2","CVD","surv_CVD")]

In [ ]:
il17r_high <- filter(data_il17rb, data_il17rb$il17rb_262 == 3)
il17r_inter <- filter(data_il17rb, data_il17rb$il17rb_262 == 2)
il17r_low <- filter(data_il17rb, data_il17rb$il17rb_262 == 1)

In [ ]:
data_il17r <- data_all_pheno_cox3[,c("ID","il17r_262","Interleukin17_receptor_B","Venous thromboembolism","baseline_age", "age2", "genetic_sex",
                                     "smoking_0", "PC1","PC2","PC3","PCD4","PC5",
                                     "chr9CN_only","VTE","mca_status","jak2_only","both_jak2_chr9cn","combine_jak2_chr9cn","surv_VTE","chip")]

In [ ]:
results <- analyze_cox_model(
  data = data_il17r, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "jak2_only", 
  covariates = c("Interleukin17_receptor_B","baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5")
)

In [ ]:
il17r_high <- filter(data_il17r, data_il17r$il17r_262 == 3)
il17r_inter <- filter(data_il17r, data_il17r$il17r_262 == 2)
il17r_low <- filter(data_il17r, data_il17r$il17r_262 == 1)

In [ ]:
results <- analyze_cox_model(
  data = il17r_high, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "jak2_only", 
  covariates = c("Interleukin17_receptor_B","baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5")
)

In [ ]:
results <- analyze_cox_model(
  data = il17r_low, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "jak2_only", 
  covariates = c("Interleukin17_receptor_B","baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5")
)

In [ ]:
results <- analyze_cox_model(
  data = il17r_inter, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "jak2_only", 
  covariates = c("Interleukin17_receptor_B","baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5")
)

#Combine_VTE

In [ ]:
results <- analyze_cox_model(
  data = il17r_high, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "jak2", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5","ldl_0","baseline_DM","BMI_0","systolicBP_0","diastolicBP_0")
)

In [ ]:
results <- analyze_cox_model(
  data = il17r_low, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "jak2", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5","ldl_0","baseline_DM","BMI_0","systolicBP_0","diastolicBP_0")
)

In [ ]:
results <- analyze_cox_model(
  data = il17r_inter, 
  time_var = "surv_VTE", 
  event_var = "VTE", 
  group_var = "jak2", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5","ldl_0","baseline_DM","BMI_0","systolicBP_0","diastolicBP_0")
)

In [ ]:
#Logistic

In [ ]:
jak2_il17r <- filter(data_il17r, data_il17r$jak2_only == 1)
dim(jak2_il17r)

In [ ]:
jak2_il17r

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = jak2_il17r,
  snp_list = "Interleukin17_receptor_B",
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
jak2_il17r_low <- filter(jak2_il17r, jak2_il17r$il17r_262 == 1)
jak2_il17r_inter <- filter(jak2_il17r, jak2_il17r$il17r_262 == 2)
jak2_il17r_high <- filter(jak2_il17r, jak2_il17r$il17r_262 == 3)

In [ ]:
table(jak2_il17r$VTE, jak2_il17r$il17r_262)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = jak2_il17r_,
  snp_list = "Interleukin17_receptor_B",
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

#CVD

In [ ]:
table(data_all_pheno_cox3$jak2)

In [ ]:
jak2_hr <- filter(data_all_pheno_cox3, data_all_pheno_cox3$jak2 == 1)

In [ ]:
analyze_cox_model0 <- function(data, time_var, event_var, group_var, covariates) {
  df_filtered <- data %>% filter(.data[[time_var]] > 0)
  df_filtered <- df_filtered %>% drop_na(all_of(covariates))
  surv_obj <- Surv(time = df_filtered[[time_var]], event = df_filtered[[event_var]] == 1)
  covariate_formula <- paste(covariates, collapse = " + ")
  formula <- as.formula(paste("surv_obj ~", group_var, "+", covariate_formula))
  cox_fit <- coxph(formula, data = df_filtered)
  summary_cox <- summary(cox_fit)
  print(summary_cox)
  results <- summary_cox$coefficients
  colnames(results) <- c("Coefficient", "Exp(Coefficient)", "Standard Error", "z value", "p value")
  print(results)
}

In [ ]:
table(jak2_hr$CVD)

In [ ]:
analyze_cox_model0(jak2_hr, "surv_CVD", "CVD", "Interleukin17_receptor_A", c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5","ldl_0","baseline_DM","BMI_0","systolicBP_0","diastolicBP_0","AID"))

In [ ]:
data_all_pheno_cox3$il17ra_333 <- 2
data_all_pheno_cox3$il17ra_333[data_all_pheno_cox3$Interleukin17_receptor_A >= quantile(data_all_pheno_cox3$Interleukin17_receptor_A,2/3)] <- 3
data_all_pheno_cox3$il17ra_333[data_all_pheno_cox3$Interleukin17_receptor_A < quantile(data_all_pheno_cox3$Interleukin17_receptor_A,1/3)] <- 1
data_all_pheno_cox3$il17ra_333<-as.factor(data_all_pheno_cox3$il17ra_333)
table(data_all_pheno_cox3$il17ra_333)

In [ ]:
table(data_all_pheno_cox3$il17ra_333,data_all_pheno_cox3$genetic_sex)

In [ ]:
data_all_pheno_cox3$AID <- NA
data_all_pheno_cox3$AID <- ifelse(data_all_pheno_cox3$`Rheumatoid arthritis`==1|
                        data_all_pheno_cox3$`Ankylosing spondylitis`==1 |
                        data_all_pheno_cox3$`Systemic lupus erythematosus [SLE]`==1|
                        data_all_pheno_cox3$Psoriasis==1|
                        data_all_pheno_cox3$`Psoriatic arthropathy`==1,1, 0)
data_all_pheno_cox3$AID <- ifelse(is.na(data_all_pheno_cox3$AID),0,data_all_pheno_cox3$AID)
table(data_all_pheno_cox3$AID, useNA = "always")

In [ ]:
data_il17rb <- data_all_pheno_cox3[,c("ID","il17ra_333","Interleukin17_receptor_B","Interleukin17_receptor_A","baseline_age", "age2", "genetic_sex",
                                     "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","baseline_DM","baseline_HTN","BMI_0","systolicBP_0","diastolicBP_0",
                                     "chip","jak2","CVD","surv_CVD","AID")]

In [ ]:
jak2_chip <- filter(data_il17rb, data_il17rb$jak2 == 1)
dim(jak2_chip)

In [ ]:
jak2_chip$il17ra_low <- 0
jak2_chip$il17ra_low[jak2_chip$Interleukin17_receptor_A < quantile(jak2_chip$Interleukin17_receptor_A,0.2)] <- 1
table(jak2_chip$il17ra_low)

In [ ]:
il17ra_high <- filter(data_il17rb, data_il17rb$il17ra_333 == 3)
il17ra_inter <- filter(data_il17rb, data_il17rb$il17ra_333 == 2)
il17ra_low <- filter(data_il17rb, data_il17rb$il17ra_333 == 1)

In [ ]:
table(il17ra_high$jak2, il17ra_high$CVD)
table(il17ra_inter$jak2, il17ra_inter$CVD)
table(il17ra_low$jak2, il17ra_low$CVD)

In [ ]:
il17ra_nonlow <- filter(data_il17rb, data_il17rb$il17ra_333 == 3 | data_il17rb$il17ra_333 == 2)

In [ ]:
results <- analyze_cox_model(
  data = il17ra_nonlow, 
  time_var = "surv_CVD", 
  event_var = "CVD", 
  group_var = "jak2", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5","ldl_0","baseline_DM","BMI_0","systolicBP_0","diastolicBP_0","AID")
)

In [ ]:
results <- analyze_cox_model(
  data = il17ra_inter, 
  time_var = "surv_CVD", 
  event_var = "CVD", 
  group_var = "jak2", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5","ldl_0","baseline_DM","BMI_0","systolicBP_0","diastolicBP_0")
)

In [ ]:
results <- analyze_cox_model(
  data = il17ra_high, 
  time_var = "surv_CVD", 
  event_var = "CVD", 
  group_var = "jak2", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5","ldl_0","baseline_DM","BMI_0","systolicBP_0","diastolicBP_0")
)

In [ ]:
results <- analyze_cox_model(
  data = il17ra_low, 
  time_var = "surv_CVD", 
  event_var = "CVD", 
  group_var = "jak2", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5","ldl_0","baseline_DM","BMI_0","systolicBP_0","diastolicBP_0","AID")
)

In [ ]:
results <- analyze_cox_model(
  data = jak2_chip, 
  time_var = "surv_CVD", 
  event_var = "CVD", 
  group_var = "il17ra_low", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5","ldl_0","baseline_DM","BMI_0","baseline_HTN","AID")
)

In [ ]:
km_fit <- survfit(Surv(surv_CVD, CVD) ~ il17ra_low, data = jak2_chip, conf.type = "log")

In [ ]:
log_rank_test <- survdiff(Surv(surv_CVD, CVD) ~ il17ra_low, data = jak2_chip)
p_value <- 1 - pchisq(log_rank_test$chisq, df = length(log_rank_test$n) - 1)

In [ ]:
p_value

In [ ]:
km_data <- tidy(km_fit)

In [ ]:
if ("strata" %in% names(km_data)) {
  km_data$group <- km_data$strata
} else {
  km_data$group <- factor(rep(levels(factor(jak2_chip$il17ra_low)), km_fit$strata))
}

In [ ]:
p <- ggplot(km_data, aes(x = time, y = estimate, color = group, fill = group)) +
  geom_step(size = 1) + 
  #geom_ribbon(aes(ymin = conf.low, ymax = conf.high), alpha = 0.2) + 
  labs(title = "Kaplan-Meier Survival Curve",
       x = "Years",
       y = "Survival Probability") +
  theme_minimal() +
  ylim(0.75, 1)
p

In [ ]:
library(sjPlot)
save_plot("kmplot_jak2_il17R.svg", fig = p, width=15, height=15)

In [ ]:
table_counts <- jak2_chip %>%
  mutate(year = floor(surv_CVD)) %>% 
  group_by(year, il17ra_low) %>%      
  summarise(count = n(), .groups = "drop") %>%  
  arrange(year, il17ra_low)         


table_wide <- table_counts %>%
  pivot_wider(names_from = il17ra_low, values_from = count, values_fill = 0) 

print(table_wide)

# CVD/MPN~ JAK2 * IL17RA

In [ ]:
save(data_all_pheno_cox3, file = "data_all_pheno_cox3_IL17R.Rdata")

In [ ]:
x <- load("data_all_pheno_cox3_IL17R.Rdata")
x

In [ ]:
table(data_all_pheno_cox3$jak2)
table(data_all_pheno_cox3$CVD)
summary(data_all_pheno_cox3$surv_CVD)
summary(data_all_pheno_cox3$Interleukin17_receptor_A)

In [ ]:
data_filter <- filter(data_all_pheno_cox3,data_all_pheno_cox3$surv_CVD >0)

In [ ]:
cox_fit <- coxph(Surv(surv_CVD,CVD)~ jak2 + Interleukin17_receptor_A + Interleukin17_receptor_A*jak2 +
                 baseline_age+age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5+ldl_0+baseline_DM+BMI_0+baseline_HTN+AID, data = data_filter)
summary(cox_fit)

In [ ]:
log <- glm(CVD ~ jak2 + Interleukin17_receptor_A + Interleukin17_receptor_A*jak2 +
                 baseline_age+age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5+
                 ldl_0+baseline_DM+BMI_0+baseline_HTN+AID, data = data_filter, family = binomial(link = "logit"))

In [ ]:
summary(log)

#sensitivity analysis

In [ ]:
table(data_filter$AID,useNA= "always")

In [ ]:
data_filter2 <- filter(data_filter,data_filter$AID  == 0)

In [ ]:
cox_fit <- coxph(Surv(surv_CVD,CVD)~ jak2 + Interleukin17_receptor_A + Interleukin17_receptor_A*jak2 +
                 baseline_age+age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5+ldl_0+baseline_DM+BMI_0+baseline_HTN, data = data_filter2)
summary(cox_fit)

#MPN

In [ ]:
data_all_pheno_cox3$MPN <- NA
data_all_pheno_cox3$MPN <- ifelse(data_all_pheno_cox3$`Myeloproliferative disorder`==1|
                        data_all_pheno_cox3$`Pulmonary embolism`==1|
                        data_all_pheno_cox3$`Venous thromboembolism`==1|
                        data_all_pheno_cox3$`Chronic myeloproliferative disease*` == 1 ,1, 0)
table(data_all_pheno_cox3$MPN)

In [ ]:
data_all_pheno_cox3$surv_MPD <- NA
data_all_pheno_cox3$surv_MPD <- as.numeric(difftime(data_all_pheno_cox3$`time_Myeloproliferative disorder`, data_all_pheno_cox3$min_date, units ="days"))/365.25
summary(data_all_pheno_cox3$surv_MPD)
data_all_pheno_cox3$surv_MPD <- ifelse(is.na(data_all_pheno_cox3$surv_MPD),data_all_pheno_cox3$surv_all,data_all_pheno_cox3$surv_MPD)
summary(data_all_pheno_cox3$surv_MPD)

In [ ]:
data_all_pheno_cox3$surv_CMD <- NA
data_all_pheno_cox3$surv_CMD <- as.numeric(difftime(data_all_pheno_cox3$`time_Chronic myeloproliferative disease*`, data_all_pheno_cox3$min_date, units ="days"))/365.25
summary(data_all_pheno_cox3$surv_CMD)
data_all_pheno_cox3$surv_CMD <- ifelse(is.na(data_all_pheno_cox3$surv_CMD),data_all_pheno_cox3$surv_all,data_all_pheno_cox3$surv_CMD)
summary(data_all_pheno_cox3$surv_CMD)

In [ ]:
data_all_pheno_cox3 <- data_all_pheno_cox3 %>% mutate(surv_MPN = pmin(surv_VTE2, surv_MPD, surv_CMD))

In [ ]:
summary(data_all_pheno_cox3$surv_MPN)

In [ ]:
data_filter <- filter(data_all_pheno_cox3,data_all_pheno_cox3$surv_MPN >0)

In [ ]:
cox_fit <- coxph(Surv(surv_MPN,MPN)~ jak2 + Interleukin17_receptor_A + Interleukin17_receptor_A*jak2 +
                 baseline_age+age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5+ldl_0+baseline_DM+BMI_0+baseline_HTN+AID, data = data_filter)
summary(cox_fit)

In [ ]:
log <- glm(MPN ~ jak2 + Interleukin17_receptor_A + Interleukin17_receptor_A*jak2 +
                 baseline_age+age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5+
                 ldl_0+baseline_DM+BMI_0+baseline_HTN+AID, data = data_filter, family = binomial(link = "logit"))
summary(log)

# JAK2 & mCA

In [ ]:
data_mca <- filter(data_all_pheno_cox2, data_all_pheno_cox2$mca_status==1)

In [ ]:
data_mca <- data_mca[,c("ID_VUMC","CVD","death","Venous thromboembolism","Pulmonary embolism","baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")]

In [ ]:
data_mca$VTE <- ifelse(is.na(data_mca$VTE),0,data_mca$VTE)
table(data_mca$VTE,useNA = "always")

In [ ]:
data_chr9_jak <- filter(data_all_pheno_cox2, data_all_pheno_cox2$ID_VUMC %in% chr9CN$ID_VUMC | data_all_pheno_cox2$ID_VUMC %in% jak2$ID_VUMC)

In [ ]:
data_chr9_jak_protein <- merge(data_chr9_jak, score_kz, by.x = "ID_VUMC" , by.y = "ID", all.x = T)

In [ ]:
dim(data_chr9_jak_protein)

In [ ]:
data_chr9_jak_protein$`Venous thromboembolism`<- ifelse(is.na(data_chr9_jak_protein$`Venous thromboembolism`),0,data_chr9_jak_protein$`Venous thromboembolism`)

In [ ]:
table(data_chr9_jak_protein$`Venous thromboembolism`)

In [ ]:
data_chr9_jak_protein$VTE <- NA
data_chr9_jak_protein$VTE <- ifelse(data_chr9_jak_protein$`Pulmonary embolism`==1|
                    data_chr9_jak_protein$`Venous thromboembolism`==1 ,1, 0)

In [ ]:
data_chr9_jak_protein$VTE<- ifelse(is.na(data_chr9_jak_protein$VTE),0,data_chr9_jak_protein$VTE)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chr9_jak_protein,
  snp_list = "Interleukin17_receptor_B",
  outcome = "VTE",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chr9_jak_protein,
  snp_list = "Interleukin17_receptor_A",
  outcome = "CVD",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_chr9_jak_protein,
  snp_list = "Interleukin17_receptor_B",
  outcome = "death",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)
logistic_mca

# IL17RB-each SNP

In [ ]:
il17r <- fread("Interleukin17_receptor_B_snp.vcf")

In [ ]:
dim(il17r)

In [ ]:
ls(il17r)

In [ ]:
sample_columns <- colnames(il17r)[10:487418]

In [ ]:
sample_columns

In [ ]:
#change version
library(tidyr)
library(dplyr)

il17r_long <- il17r %>%
  pivot_longer(
    cols = all_of(sample_columns),  
    names_to = "Sample",           
    values_to = "Genotype"        
  )

In [ ]:
il17r_long <- il17r_long[,c(3,10,11)]

In [ ]:
il17r_wide <- il17r_long %>%
  pivot_wider(
    names_from = ID,      
    values_from = Genotype 
  )

In [ ]:
il17r_wide <- il17r_wide %>%
  mutate(across(all_of(target_columns), ~ case_when(
    . == "0/0" ~ 0,
    . == "0/1" ~ 1,
    . == "1/1" ~ 2,
    TRUE ~ NA_real_  
  )))

In [ ]:
il17r_wide <- il17r_wide %>%
  mutate(Sample = sub("_.*", "", Sample))

In [ ]:
write.csv(il17r_wide,file= "il17r_SNP_data_UKB.csv")

In [ ]:
il17r_wide <- read.csv("il17r_SNP_data_UKB.csv")

In [ ]:
target_columns <- colnames(il17r_wide)[3:53]
target_columns

In [ ]:
#Load UKB
x <- load("UKB_data_all_1024.Rdata")
x

In [ ]:
data_chip2$ID_VUMC <- as.character(data_chip2$ID_VUMC)
il17r_wide$Sample <- as.character(il17r_wide$Sample)

In [ ]:
data_chip3 <- merge(data_chip2,il17r_wide,by.x = "ID_VUMC", by.y = "Sample", all = F)

In [ ]:
ls(data_chip3)

In [ ]:
data_chip3$jak2 <- ifelse(data_chip3$ID_VUMC %in% jak2$ID_VUMC , 1, 0)

In [ ]:
data_jak2 <- filter(data_chip3, data_chip3$jak2 == 1)

In [ ]:
dim(data_jak2)

#lasso

In [ ]:
library(glmnet)

In [ ]:
outcome <- data_jak2$"`Venous thromboembolism`"
snp_matrix <- as.matrix(data_jak2[, ..target_columns]) 
snp_matrix <- apply(snp_matrix, 2, as.numeric) 

In [ ]:
fill_missing_with_mode <- function(df) {
  apply(df, 2, function(column) {
    if (any(is.na(column))) {
      mode_value <- as.numeric(names(which.max(table(column, useNA = "no"))))
      column[is.na(column)] <- mode_value
    }
    return(column)
  })
}

snp_matrix_filled <- fill_missing_with_mode(snp_matrix)

snp_matrix_filled <- as.matrix(snp_matrix_filled)

sum(is.na(snp_matrix_filled))

In [ ]:
X <- as.matrix(data_jak2[, ..target_columns]) 
y <- data_jak2$`Venous thromboembolism`
covariates <- as.matrix(data_jak2[, c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","systolicBP_0","diastolicBP_0","baseline_DM")])

X_combined <- cbind(covariates, X)

In [ ]:
fill_missing_with_mode <- function(df) {
  apply(df, 2, function(column) {
    if (any(is.na(column))) {
      mode_value <- as.numeric(names(which.max(table(column, useNA = "no"))))
      column[is.na(column)] <- mode_value
    }
    return(column)
  })
}

X_combined <- fill_missing_with_mode(X_combined)

sum(is.na(X_combined)) 

In [ ]:
penalty_factors <- c(rep(0, ncol(covariates)), rep(1, ncol(X)))

lasso_model <- cv.glmnet(
  x = X_combined,
  y = y,
  alpha = 1,  
  family = "binomial",  
  penalty.factor = penalty_factors  
)

best_lambda <- lasso_model$lambda.min
coef(lasso_model, s = best_lambda)

In [ ]:
library(pROC)
predictions <- predict(lasso_model, newx = X_combined, s = best_lambda, type = "response")
roc_curve <- roc(data_jak2$`Venous thromboembolism`, predictions)
auc(roc_curve)

In [ ]:
plot(roc_curve, main = "LASSO Model ROC Curve")

In [ ]:
predicted_probs <- predict(lasso_model, newx = X_combined, s = best_lambda, type = "response")

In [ ]:
if (!requireNamespace("PRROC", quietly = TRUE)) {
  install.packages("PRROC")
}
library(PRROC)

pr_curve <- pr.curve(scores.class0 = predicted_probs, weights.class0 = y, curve = TRUE)

cat("Precision-Recall AUC:", pr_curve$auc.integral, "\n")

plot(pr_curve)

In [ ]:
if (!requireNamespace("precrec", quietly = TRUE)) {
  install.packages("precrec")
}
library(precrec)
pr <- evalmod(scores = predicted_probs, labels = y)

autoplot(pr)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = target_columns,
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")
)

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

In [ ]:
table(data_jak2$`Venous thromboembolism`)
table(data_jak2$CVD)

In [ ]:
logistic_mca <- batch_logistic_regression(
  data = data_jak2,
  snp_list = target_columns,
  outcome = "`Venous thromboembolism`",
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","baseline_HTN","baseline_DM")
)

In [ ]:
logistic_mca <- logistic_mca %>% arrange(P_Value)
logistic_mca

# IL17RA

In [ ]:
il17ra <- fread("Interleukin17_receptor_A_snp.vcf")

In [ ]:
dim(il17ra)

In [ ]:
sample_columns <- colnames(il17ra)[10:487418]
sample_columns

In [ ]:
#change version
library(tidyr)
library(dplyr)

il17r_long <- il17ra %>%
  pivot_longer(
    cols = all_of(sample_columns),  
    names_to = "Sample",           
    values_to = "Genotype"        
  )

In [ ]:
il17r_long <- il17r_long[,c(3,10,11)]

In [ ]:
il17r_wide <- il17r_long %>%
  pivot_wider(
    names_from = ID,      
    values_from = Genotype 
  )

In [ ]:
il17r_wide <- il17r_wide %>%
  mutate(across(all_of(target_columns), ~ case_when(
    . == "0/0" ~ 0,
    . == "0/1" ~ 1,
    . == "1/1" ~ 2,
    TRUE ~ NA_real_  
  )))

In [ ]:
il17r_wide <- il17r_wide %>%
  mutate(Sample = sub("_.*", "", Sample))

In [ ]:
il17r_wide

In [ ]:
write.csv(il17r_wide,file= "il17ra_SNP_data_UKB.csv")

In [ ]:
il17ra_wide <- read.csv("il17ra_SNP_data_UKB.csv")

In [ ]:
target_columns_a <- colnames(il17ra_wide)[3:78]
target_columns_a
length(target_columns_a)

In [ ]:
data_chip2$ID_VUMC <- as.character(data_chip2$ID_VUMC)
il17ra_wide$Sample <- as.character(il17ra_wide$Sample)
data_chip3a <- merge(data_chip2,il17ra_wide,by.x = "ID_VUMC", by.y = "Sample", all = F)

In [ ]:
ls(data_chip3a)

In [ ]:
data_chip3a$jak2 <- ifelse(data_chip3a$ID_VUMC %in% jak2$ID_VUMC , 1, 0)
data_jak2a <- filter(data_chip3a, data_chip3a$jak2 == 1)
dim(data_jak2a)

#lasso

In [ ]:
library(glmnet)

In [ ]:
outcome <- data_jak2a$CVD 
snp_matrix <- as.matrix(data_jak2a[, ..target_columns_a]) 
snp_matrix <- apply(snp_matrix, 2, as.numeric) 

In [ ]:
fill_missing_with_mode <- function(df) {
  apply(df, 2, function(column) {
    if (any(is.na(column))) {
      mode_value <- as.numeric(names(which.max(table(column, useNA = "no"))))
      column[is.na(column)] <- mode_value
    }
    return(column)
  })
}

snp_matrix_filled <- fill_missing_with_mode(snp_matrix)

snp_matrix_filled <- as.matrix(snp_matrix_filled)

sum(is.na(snp_matrix_filled))

In [ ]:
X <- as.matrix(data_jak2a[, ..target_columns_a]) 
y <- data_jak2a$CVD  
covariates <- as.matrix(data_jak2a[, c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5","ldl_0","BMI_0","systolicBP_0","diastolicBP_0","baseline_DM")])

X_combined <- cbind(covariates, X)

In [ ]:
fill_missing_with_mode <- function(df) {
  apply(df, 2, function(column) {
    if (any(is.na(column))) {
      mode_value <- as.numeric(names(which.max(table(column, useNA = "no"))))
      column[is.na(column)] <- mode_value
    }
    return(column)
  })
}

X_combined <- fill_missing_with_mode(X_combined)

sum(is.na(X_combined)) 

In [ ]:
penalty_factors <- c(rep(0, ncol(covariates)), rep(1, ncol(X)))

lasso_model <- cv.glmnet(
  x = X_combined,
  y = y,
  alpha = 1,  
  family = "binomial",  
  penalty.factor = penalty_factors  
)

best_lambda <- lasso_model$lambda.min
coef(lasso_model, s = best_lambda)

In [ ]:
library("pROC")
predictions <- predict(lasso_model, newx = X_combined, s = best_lambda, type = "response")
roc_curve <- roc(data_jak2a$CVD, predictions)
auc(roc_curve)

In [ ]:
plot(roc_curve, main = "LASSO Model ROC Curve")

In [ ]:
predicted_probs <- predict(lasso_model, newx = X_combined, s = best_lambda, type = "response")

In [ ]:
library(PRROC)

pr_curve <- pr.curve(scores.class0 = predicted_probs, weights.class0 = y, curve = TRUE)

cat("Precision-Recall AUC:", pr_curve$auc.integral, "\n")

plot(pr_curve)

In [ ]:
library(precrec)
pr <- evalmod(scores = predicted_probs, labels = y)

autoplot(pr)

In [ ]:
system("gsutil cp gs://bicklab-main-storage/Users/Caitlyn_Vlasschaert/demographic_data_Aug15_participant.tsv .", intern = TRUE)

In [ ]:
data_covari <- fread('demographic_data_Aug15_participant.tsv')

In [ ]:
ls(data_covari)
dim(data_covari)

# rs17425819 & VTE

In [ ]:
system("gsutil cp gs://bicklab-main-storage/Users/Yash_Pershad/ukb_interaction_hit_dataframes_for_plots_07192024/jak2_plt_interaction_plot_data.tsv .", intern = TRUE)

In [ ]:
jak2_germ <- fread("jak2_plt_interaction_plot_data.tsv")

In [ ]:
dim(jak2_germ)
jak2_germ

In [ ]:
table(jak2_germ$rs17425819, useNA = "always")
ls(jak2_germ)

In [ ]:
jak2_germ <- jak2_germ[,c("ID_VUMC","rs17425819")]

In [ ]:
jak2_germ$ID_VUMC <- as.character(jak2_germ$ID_VUMC)

In [ ]:
data_il17rb_germ <- merge(jak2_germ,data_il17rb,by.x = "ID_VUMC",by.y = "ID", all = F)

In [ ]:
dim(data_il17rb_germ)
ls(data_il17rb_germ)

In [ ]:
data_il17rb_germ$rs17425819_score <- data_il17rb_germ$rs17425819* data_il17rb_germ$Interleukin17_receptor_B

In [ ]:
summary(data_il17rb_germ$rs17425819_score)

In [ ]:
table(data_il17rb_germ$`Venous thromboembolism`,useNA = "always")
summary(data_il17rb_germ$surv_VTE)
summary(data_il17rb_germ$surv_VTE2)

In [ ]:
table(data_il17rb_germ$VTE)
table(data_il17rb_germ$`Venous thromboembolism`)

In [ ]:
data_il17rb_germ$VenousThromboEmbolism <- data_il17rb_germ$`Venous thromboembolism`

In [ ]:
results <- analyze_cox_model(
  data = data_il17rb_germ, 
  time_var = "surv_VTE", 
  event_var = "VenousThromboEmbolism", 
  group_var = "rs17425819*Interleukin17_receptor_B", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5", "ldl_0","baseline_DM","BMI_0","systolicBP_0","diastolicBP_0")
)

In [ ]:
data_il17rb_jak2 <- filter(data_il17rb_germ, data_il17rb_germ$jak2 == 1)

In [ ]:
dim(data_il17rb_jak2)

In [ ]:
results <- analyze_cox_model(
  data = data_il17rb_jak2, 
  time_var = "surv_VTE", 
  event_var = "VenousThromboEmbolism", 
  group_var = "rs17425819*Interleukin17_receptor_B", 
  covariates = c("baseline_age", "age2", "genetic_sex", "smoking_0", "PC1", "PC2", "PC3", "PCD4", "PC5", "ldl_0","baseline_DM","BMI_0","systolicBP_0","diastolicBP_0")
)

In [ ]:
logistic_model <- glm(`Venous thromboembolism`~Interleukin17_receptor_B*rs17425819 + baseline_age+age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5
                      +ldl_0+BMI_0+systolicBP_0+diastolicBP_0+baseline_DM, data = data_il17rb_germ, family = binomial)
summary(logistic_model)

In [ ]:
logistic_model <- glm(`Venous thromboembolism`~Interleukin17_receptor_B*rs17425819 + baseline_age+age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5+ldl_0+BMI_0+systolicBP_0+diastolicBP_0+baseline_DM, data = data_il17rb_jak2, family = binomial)
summary(logistic_model)

# TEST t2e CHIP CVD

In [ ]:
jak2_germ <- fread("jak2_plt_interaction_plot_data.tsv")
dim(jak2_germ)
jak2_germ

In [ ]:
jak2_germ <- jak2_germ[,c(2:7,10)]

In [ ]:
t2e_chip <- read.csv("ukb_chip_cvd_t2e_clean.csv")

In [ ]:
dim(t2e_chip)
head(t2e_chip)

In [ ]:
t2e_chip_all <- merge(t2e_chip,jak2_germ, by.x = "FID", by.y = "ID_VUMC", all.x = T)

In [ ]:
dim(t2e_chip_all)
ls(t2e_chip_all)

In [ ]:
cox <- coxph(Surv(surv_CVDcensor, CVD) ~rs4903127 + baseline_age + genetic_sex +PC1+PC2+PC3+PCD4+PC5+PC6+PC7+PC8+
             PC9+PC10+ baseline_DM + baseline_HTN + ldl_0, data = t2e_chip_all)

In [ ]:
summary(cox)

# Protein Measurement

In [ ]:
score_yp <- fread("ukb_52k_proteome_dec12_yp.tsv")

In [ ]:
dim(score_yp)

In [ ]:
0.05/1465

In [ ]:
data_ukb2 <- merge(data_all_pheno_cox2, score_yp, by.x = "ID_VUMC", by.y = "eid", all = F)

#define m-CH and l-CH

In [ ]:
table(data_all_pheno_cox2$mca_status)

In [ ]:
calls_mca <- fread('mocha_ukb_autosomal_mca_calls.tsv') #17865
head(calls_mca)

In [ ]:
#mCH
mca_df <- calls_mca
mca_df$mCH<-NA
mca_df$mCH<-ifelse(mca_df$type=="Loss" & mca_df$chrom =="chr12" & mca_df$q_arm !="N",1,mca_df$mCH)
mca_df$mCH<-ifelse(mca_df$type=="Loss" & mca_df$chrom =="chr20" & mca_df$q_arm !="N",1,mca_df$mCH)
mca_df$mCH<-ifelse(mca_df$type=="Loss" & mca_df$chrom =="chr5" & mca_df$q_arm !="N",1,mca_df$mCH)
mca_df$mCH<-ifelse(mca_df$type=="Gain" & mca_df$chrom =="chr1" & mca_df$q_arm !="N",1,mca_df$mCH)
mca_df$mCH<-ifelse(mca_df$type=="Gain" & mca_df$chrom =="chr9" & mca_df$p_arm !="N",1,mca_df$mCH)
mca_df$mCH<-ifelse(mca_df$type=="CN-LOH" & mca_df$chrom =="chr22" & mca_df$q_arm !="N",1,mca_df$mCH)
mca_df$mCH<-ifelse(mca_df$type=="CN-LOH" & mca_df$chrom =="chr9" & mca_df$p_arm !="N",1,mca_df$mCH)
mca_df$mCH<-ifelse(mca_df$type=="CN-LOH" & mca_df$chrom =="chr14" & mca_df$q_arm !="N",1,mca_df$mCH)
mca_df$mCH<-ifelse(mca_df$type=="Gain" & mca_df$chrom =="chr8" & mca_df$q_arm =="Y" & mca_df$p_arm =="Y",1,mca_df$mCH)
mca_df$mCH<-ifelse(mca_df$type=="Gain" & mca_df$chrom =="chr8" & mca_df$q_arm =="T" & mca_df$p_arm =="T",1,mca_df$mCH)
mca_df$mCH<-ifelse(mca_df$type=="Gain" & mca_df$chrom =="chr8" & mca_df$q_arm =="Y" & mca_df$p_arm =="T",1,mca_df$mCH)
mca_df$mCH<-ifelse(mca_df$type=="Gain" & mca_df$chrom =="chr8" & mca_df$q_arm =="T" & mca_df$p_arm =="Y",1,mca_df$mCH)
table(mca_df$mCH)

In [ ]:
#lCH
mca_df$lCH<-NA
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr10" & mca_df$p_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr10" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr11" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr13" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr14" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr15" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr17" & mca_df$p_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr1" & mca_df$p_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr1" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr22" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr6" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr7" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Loss" & mca_df$chrom == "chr8" & mca_df$p_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr12" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr15" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr17" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr22" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr2" & mca_df$p_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr3" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr8" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr9" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "CN-LOH" & mca_df$chrom == "chr16" & mca_df$p_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "CN-LOH" & mca_df$chrom == "chr1" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "CN-LOH" & mca_df$chrom == "chr7" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "CN-LOH" & mca_df$chrom == "chr13" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "CN-LOH" & mca_df$chrom == "chr12" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "CN-LOH" & mca_df$chrom == "chr9" & mca_df$q_arm != "N", 1, mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr12" & mca_df$q_arm =="Y" & mca_df$p_arm =="Y",1,mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr12" & mca_df$q_arm =="T" & mca_df$p_arm =="Y",1,mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr12" & mca_df$q_arm =="Y" & mca_df$p_arm =="T",1,mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr12" & mca_df$q_arm =="T" & mca_df$p_arm =="T",1,mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr18" & mca_df$q_arm =="Y" & mca_df$p_arm =="Y",1,mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr18" & mca_df$q_arm =="T" & mca_df$p_arm =="Y",1,mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr18" & mca_df$q_arm =="Y" & mca_df$p_arm =="T",1,mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr18" & mca_df$q_arm =="T" & mca_df$p_arm =="T",1,mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr19" & mca_df$q_arm =="Y" & mca_df$p_arm =="Y",1,mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr19" & mca_df$q_arm =="T" & mca_df$p_arm =="Y",1,mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr19" & mca_df$q_arm =="Y" & mca_df$p_arm =="T",1,mca_df$lCH)
mca_df$lCH <- ifelse(mca_df$type == "Gain" & mca_df$chrom == "chr19" & mca_df$q_arm =="T" & mca_df$p_arm =="T",1,mca_df$lCH)
table(mca_df$lCH)

In [ ]:
mch_person_ids <- mca_df %>%
  filter(mCH == 1) %>%
  pull(ID_VUMC)
lch_person_ids <- mca_df %>%
  filter(lCH == 1) %>%
  pull(ID_VUMC)

In [ ]:
data_ukb2$lCH<-NA
data_ukb2 <- data_ukb2 %>%
  mutate(lCH = ifelse(ID_VUMC %in% lch_person_ids, 1, 0))

In [ ]:
data_ukb2$mCH<-NA
data_ukb2 <- data_ukb2 %>%
  mutate(mCH = ifelse(ID_VUMC %in% mch_person_ids, 1, 0))

In [ ]:
table(data_ukb2$mCH)
table(data_ukb2$lCH)
table(data_ukb2$mca_status)

In [ ]:
proteinlist <- colnames(score_yp)[3:1465] #3:1465
proteinlist

In [ ]:
result_list <- lapply(proteinlist, function(protein) {
  formula <- as.formula(paste(protein, "~ mca_status + baseline_age + age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5"))  
  model <- lm(formula, data = data_ukb2)                          
  summary_model <- summary(model)
  
  mca_row <- coef(summary_model)["mca_status", ]
  data.frame(
    Protein = protein,
    Beta = mca_row["Estimate"],
    SE = mca_row["Std. Error"],
    P_Value = mca_row["Pr(>|t|)"]
  )
})


result_df <- do.call(rbind, result_list)
print(result_df)

In [ ]:
result_df <- as.data.frame(result_df)

In [ ]:
result_df <- result_df %>% arrange(P_Value)
result_df

In [ ]:
result_df <- result_df %>% arrange(Protein)
write.csv(result_df,file="Protein_mCAs.csv")

In [ ]:
install.packages("sjPlot")
library(sjPlot)

In [ ]:
result_df$LogP <- -log10(result_df$P_Value)
result_df$Significant <- ifelse(result_df$LogP > 5, TRUE, FALSE)
result_df$Label <- ifelse(result_df$LogP > 20 | result_df$Beta > 0.15, result_df$Protein, "")

p <- ggplot(result_df, aes(x = Beta, y = LogP, label = Label)) +
  geom_point(aes(color = Significant), alpha = 0.7, size = 3) +  
  scale_color_manual(values = c("black", "red")) +              
  geom_text_repel(                                              
    box.padding = 0.35, 
    point.padding = 0.3, 
    max.overlaps = 10, 
    size = 4, 
    segment.color = "grey50"
  ) +
  theme_minimal(base_size = 14) +                             
  labs(
    x = "Beta (Estimate)", 
    y = "-log10(P-value)", 
    title = "Enhanced Volcano Plot",
    caption = "Points in red: LogP > 5; Labels: LogP > 20"
  ) +
  theme(
    plot.title = element_text(size = 18, face = "bold", hjust = 0.5),  
    plot.caption = element_text(size = 12, face = "italic", hjust = 0.5),  
    legend.position = "none"                            
  ) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05))) +         
  scale_x_continuous(expand = expansion(mult = c(0.05, 0.05))) 
p

In [ ]:
save_plot("volcano_protein_mca.svg", fig = p, width=20, height=16)

In [ ]:
sig_mca <- filter(result_df, result_df$LogP > 4.5)
dim(sig_mca)

In [ ]:
sig_mca$Protein

In [ ]:
result_list <- lapply(proteinlist, function(protein) {
  formula <- as.formula(paste(protein, "~ mCH + baseline_age + age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5"))  
  model <- lm(formula, data = data_ukb2)                          
  summary_model <- summary(model)
  
  mca_row <- coef(summary_model)["mCH", ]
  data.frame(
    Protein = protein,
    Beta = mca_row["Estimate"],
    SE = mca_row["Std. Error"],
    P_Value = mca_row["Pr(>|t|)"]
  )
})


result_mCH <- do.call(rbind, result_list)
print(result_mCH)

In [ ]:
result_mCH <- as.data.frame(result_mCH)
result_mCH <- result_mCH %>% arrange(P_Value)
result_mCH

In [ ]:
result_mCH$LogP <- -log10(result_mCH$P_Value)

threshold <- 10  
result_mCH$Significant <- ifelse(result_mCH$LogP > threshold, TRUE, FALSE)


ggplot(result_mCH, aes(x = Beta, y = LogP, label = ifelse(Significant, Protein, ""))) +
  geom_point(aes(color = Significant), alpha = 0.8) +  
  scale_color_manual(values = c("black", "red")) +     
  geom_text_repel(box.padding = 0.3, max.overlaps = 10) +  
  theme_minimal() +
  labs(
    x = "Estimate",
    y = "-log10(P-value)",
    title = "Volcano Plot"
  ) +
  theme(
    legend.position = "none"
  )

In [ ]:
#lCH
result_list <- lapply(proteinlist, function(protein) {
  formula <- as.formula(paste(protein, "~ lCH + baseline_age + age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5"))  
  model <- lm(formula, data = data_ukb2)                          
  summary_model <- summary(model)
  
  mca_row <- coef(summary_model)["lCH", ]
  data.frame(
    Protein = protein,
    Beta = mca_row["Estimate"],
    SE = mca_row["Std. Error"],
    P_Value = mca_row["Pr(>|t|)"]
  )
})


result_lCH <- do.call(rbind, result_list)
result_lCH <- as.data.frame(result_lCH)
result_lCH <- result_lCH %>% arrange(P_Value)
result_lCH

In [ ]:
result_lCH <- result_lCH %>% arrange(Protein)
write.csv(result_lCH,file="Protein_l_mCAs.csv")

In [ ]:
result_lCH$LogP <- -log10(result_lCH$P_Value)
result_lCH$Significant <- ifelse(result_lCH$LogP > 5, TRUE, FALSE)
result_lCH$Label <- ifelse(result_lCH$LogP > 20 | result_lCH$Beta > 0.3, result_lCH$Protein, "")

p <- ggplot(result_lCH, aes(x = Beta, y = LogP, label = Label)) +
  geom_point(aes(color = Significant), alpha = 0.7, size = 3) +  
  scale_color_manual(values = c("black", "red")) +              
  geom_text_repel(                                              
    box.padding = 0.35, 
    point.padding = 0.3, 
    max.overlaps = 10, 
    size = 4, 
    segment.color = "grey50"
  ) +
  theme_minimal(base_size = 14) +                             
  labs(
    x = "Beta (Estimate)", 
    y = "-log10(P-value)", 
    title = "Enhanced Volcano Plot",
    caption = "Points in red: LogP > 5; Labels: LogP > 20"
  ) +
  theme(
    plot.title = element_text(size = 18, face = "bold", hjust = 0.5),  
    plot.caption = element_text(size = 12, face = "italic", hjust = 0.5),  
    legend.position = "none"                            
  ) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05))) +         
  scale_x_continuous(expand = expansion(mult = c(0.05, 0.05))) 
p

In [ ]:
save_plot("volcano_protein_lmca.svg", fig = p, width=20, height=16)

In [ ]:
sig_l_mca <- filter(result_lCH, result_lCH$LogP > 4.5)
dim(sig_l_mca)

In [ ]:
#high risk CLL mCA

In [ ]:
table(data_ukb2$mca_highrisk)

In [ ]:
data_ukb_highrisk <- filter(data_ukb2, data_ukb2$mca_highrisk == 0 | data_ukb2$mca_highrisk == 1)

In [ ]:
data_ukb_highrisk$mca_highrisk2 <- NA
data_ukb_highrisk$mca_highrisk2 <- ifelse(data_ukb_highrisk$mca_highrisk == 0, 1 ,2)

In [ ]:
table(data_ukb_highrisk$mca_highrisk2)

In [ ]:
#high risk 
result_list <- lapply(proteinlist, function(protein) {
  formula <- as.formula(paste(protein, "~ mca_highrisk2 + baseline_age + age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5"))  
  model <- lm(formula, data = data_ukb_highrisk)                          
  summary_model <- summary(model)
  
  mca_row <- coef(summary_model)["mca_highrisk2", ]
  data.frame(
    Protein = protein,
    Beta = mca_row["Estimate"],
    SE = mca_row["Std. Error"],
    P_Value = mca_row["Pr(>|t|)"]
  )
})


result_highrisk <- do.call(rbind, result_list)
result_highrisk <- as.data.frame(result_highrisk)
result_highrisk <- result_highrisk %>% arrange(P_Value)
result_highrisk

In [ ]:
result_highrisk <- result_highrisk %>% arrange(Protein)
write.csv(result_highrisk,file="Protein_highrisk_mCAs.csv")

In [ ]:
result_highrisk$LogP <- -log10(result_highrisk$P_Value)
result_highrisk$Significant <- ifelse(result_highrisk$LogP > 5, TRUE, FALSE)
result_highrisk$Label <- ifelse(result_highrisk$LogP > 20 | result_highrisk$Beta > 0.5, result_highrisk$Protein, "")

p <- ggplot(result_highrisk, aes(x = Beta, y = LogP, label = Label)) +
  geom_point(aes(color = Significant), alpha = 0.7, size = 3) +  
  scale_color_manual(values = c("black", "red")) +              
  geom_text_repel(                                              
    box.padding = 0.35, 
    point.padding = 0.3, 
    max.overlaps = 10, 
    size = 4, 
    segment.color = "grey50"
  ) +
  theme_minimal(base_size = 14) +                             
  labs(
    x = "Beta (Estimate)", 
    y = "-log10(P-value)", 
    title = "Enhanced Volcano Plot",
    caption = "Points in red: LogP > 5; Labels: LogP > 20"
  ) +
  theme(
    plot.title = element_text(size = 18, face = "bold", hjust = 0.5),  
    plot.caption = element_text(size = 12, face = "italic", hjust = 0.5),  
    legend.position = "none"                            
  ) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05))) +         
  scale_x_continuous(expand = expansion(mult = c(0.05, 0.05))) 
p

In [ ]:
save_plot("volcano_protein_highriskmca.svg", fig = p, width=20, height=16)

#CF-mCA

In [ ]:
dim(data_ukb2)

In [ ]:
data_ukb_mca <- filter(data_ukb2, data_ukb2$mca_status == 1)

In [ ]:
dim(data_ukb_mca)

In [ ]:
summary(data_ukb_mca$cf_max)

In [ ]:
result_list <- lapply(proteinlist, function(protein) {
  formula <- as.formula(paste(protein, "~ cf_max + baseline_age + age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5"))  
  model <- lm(formula, data = data_ukb_mca)                          
  summary_model <- summary(model)
  
  mca_row <- coef(summary_model)["cf_max", ]
  data.frame(
    Protein = protein,
    Beta = mca_row["Estimate"],
    SE = mca_row["Std. Error"],
    P_Value = mca_row["Pr(>|t|)"]
  )
})


In [ ]:
result_cf_mca <- do.call(rbind, result_list)
result_cf_mca <- as.data.frame(result_cf_mca)
result_cf_mca <- result_cf_mca %>% arrange(P_Value)
result_cf_mca

In [ ]:
result_cf_mca$LogP <- -log10(result_cf_mca$P_Value)
result_cf_mca$Significant <- ifelse(result_cf_mca$LogP > 5, TRUE, FALSE)
result_cf_mca$Label <- ifelse(result_cf_mca$LogP > 20 | result_cf_mca$Beta > 1.5, result_cf_mca$Protein, "")

p <- ggplot(result_cf_mca, aes(x = Beta, y = LogP, label = Label)) +
  geom_point(aes(color = Significant), alpha = 0.7, size = 3) +  
  scale_color_manual(values = c("black", "red")) +              
  geom_text_repel(                                              
    box.padding = 0.35, 
    point.padding = 0.3, 
    max.overlaps = 10, 
    size = 4, 
    segment.color = "grey50"
  ) +
  theme_minimal(base_size = 14) +                             
  labs(
    x = "Beta (Estimate)", 
    y = "-log10(P-value)", 
    title = "Enhanced Volcano Plot",
    caption = "Points in red: LogP > 5; Labels: LogP > 20"
  ) +
  theme(
    plot.title = element_text(size = 18, face = "bold", hjust = 0.5),  
    plot.caption = element_text(size = 12, face = "italic", hjust = 0.5),  
    legend.position = "none"                            
  ) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05))) +         
  scale_x_continuous(expand = expansion(mult = c(0.05, 0.05))) 
p

In [ ]:
save_plot("volcano_protein_CF.svg", fig = p, width=20, height=16)

In [ ]:
result_cf_mca <- result_cf_mca %>% arrange(Protein)
write.csv(result_cf_mca,file="Protein_CF_mCAs.csv")

#CF-lmCA

In [ ]:
data_ukb_lmca <- filter(data_ukb2, data_ukb2$lCH == 1)

In [ ]:
dim(data_ukb_lmca)

In [ ]:
result_list <- lapply(proteinlist, function(protein) {
  formula <- as.formula(paste(protein, "~ cf_max + baseline_age + age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5"))  
  model <- lm(formula, data = data_ukb_lmca)                          
  summary_model <- summary(model)
  
  mca_row <- coef(summary_model)["cf_max", ]
  data.frame(
    Protein = protein,
    Beta = mca_row["Estimate"],
    SE = mca_row["Std. Error"],
    P_Value = mca_row["Pr(>|t|)"]
  )
})


In [ ]:
result_cf_lmca <- do.call(rbind, result_list)
result_cf_lmca <- as.data.frame(result_cf_lmca)
result_cf_lmca <- result_cf_lmca %>% arrange(P_Value)
result_cf_lmca

In [ ]:
result_cf_lmca$LogP <- -log10(result_cf_lmca$P_Value)
result_cf_lmca$Significant <- ifelse(result_cf_lmca$LogP > 5, TRUE, FALSE)
result_cf_lmca$Label <- ifelse(result_cf_lmca$LogP > 21 | result_cf_lmca$Beta > 3, result_cf_lmca$Protein, "")

p <- ggplot(result_cf_lmca, aes(x = Beta, y = LogP, label = Label)) +
  geom_point(aes(color = Significant), alpha = 0.7, size = 3) +  
  scale_color_manual(values = c("black", "red")) +              
  geom_text_repel(                                              
    box.padding = 0.35, 
    point.padding = 0.3, 
    max.overlaps = 10, 
    size = 4, 
    segment.color = "grey50"
  ) +
  theme_minimal(base_size = 14) +                             
  labs(
    x = "Beta (Estimate)", 
    y = "-log10(P-value)", 
    title = "Enhanced Volcano Plot",
    caption = "Points in red: LogP > 5; Labels: LogP > 20"
  ) +
  theme(
    plot.title = element_text(size = 18, face = "bold", hjust = 0.5),  
    plot.caption = element_text(size = 12, face = "italic", hjust = 0.5),  
    legend.position = "none"                            
  ) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05))) +         
  scale_x_continuous(expand = expansion(mult = c(0.05, 0.05))) 
p

In [ ]:
save_plot("volcano_protein_CF_lmCA.svg", fig = p, width=20, height=16)

In [ ]:
result_cf_lmca <- result_cf_lmca %>% arrange(Protein)
write.csv(result_cf_lmca,file="Protein_CF_lmCAs.csv")

#CF-highrisk

In [ ]:
data_ukb_highrisk <- filter(data_ukb2, data_ukb2$mca_highrisk == 1)

In [ ]:
dim(data_ukb_highrisk)

In [ ]:
result_list <- lapply(proteinlist, function(protein) {
  formula <- as.formula(paste(protein, "~ cf_max + baseline_age + age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5"))  
  model <- lm(formula, data = data_ukb_highrisk)                          
  summary_model <- summary(model)
  
  mca_row <- coef(summary_model)["cf_max", ]
  data.frame(
    Protein = protein,
    Beta = mca_row["Estimate"],
    SE = mca_row["Std. Error"],
    P_Value = mca_row["Pr(>|t|)"]
  )
})


In [ ]:
result_cf_highr <- do.call(rbind, result_list)
result_cf_highr <- as.data.frame(result_cf_highr)
result_cf_highr <- result_cf_highr %>% arrange(P_Value)
result_cf_highr

In [ ]:
result_cf_highr$LogP <- -log10(result_cf_highr$P_Value)
result_cf_highr$Significant <- ifelse(result_cf_highr$LogP > 5, TRUE, FALSE)
result_cf_highr$Label <- ifelse(result_cf_highr$LogP > 15 | result_cf_highr$Beta > 3, result_cf_highr$Protein, "")

p <- ggplot(result_cf_highr, aes(x = Beta, y = LogP, label = Label)) +
  geom_point(aes(color = Significant), alpha = 0.7, size = 3) +  
  scale_color_manual(values = c("black", "red")) +              
  geom_text_repel(                                              
    box.padding = 0.35, 
    point.padding = 0.3, 
    max.overlaps = 10, 
    size = 4, 
    segment.color = "grey50"
  ) +
  theme_minimal(base_size = 14) +                             
  labs(
    x = "Beta (Estimate)", 
    y = "-log10(P-value)", 
    title = "Enhanced Volcano Plot",
    caption = "Points in red: LogP > 5; Labels: LogP > 20"
  ) +
  theme(
    plot.title = element_text(size = 18, face = "bold", hjust = 0.5),  
    plot.caption = element_text(size = 12, face = "italic", hjust = 0.5),  
    legend.position = "none"                            
  ) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05))) +         
  scale_x_continuous(expand = expansion(mult = c(0.05, 0.05))) 
p

In [ ]:
save_plot("volcano_protein_CF_highrisk.svg", fig = p, width=20, height=16)

In [ ]:
result_cf_highr <- result_cf_highr %>% arrange(Protein)
write.csv(result_cf_highr,file="Protein_CF_hr_mCAs.csv")

# Genetically predicted proteins and mCAs (Quit)

In [ ]:
head(data_all_pheno_cox2)

In [ ]:
data_clean <- data_all_pheno_cox2[,c("ID_VUMC","mca_status","baseline_age", "age2", "genetic_sex", "smoking_0", "PC1","PC2","PC3","PCD4","PC5")]

In [ ]:
data_clean2 <- merge(data_clean, score_kz, by.x = "ID_VUMC", by.y = "ID", all = F)

In [ ]:
dim(data_clean2)

In [ ]:
data_clean2$lCH<-NA
data_clean2 <- data_clean2 %>%
  mutate(lCH = ifelse(ID_VUMC %in% lch_person_ids, 1, 0))

data_clean2$mCH<-NA
data_clean2 <- data_clean2 %>%
  mutate(mCH = ifelse(ID_VUMC %in% mch_person_ids, 1, 0))

In [ ]:
table(data_clean2$mCH)
table(data_clean2$lCH)
table(data_clean2$mca_status)

In [ ]:
pro_list <- colnames(score_kz)[-1]

In [ ]:
result_list <- lapply(pro_list, function(protein) {
  formula <- as.formula(paste(protein, "~ mca_status + baseline_age + age2+genetic_sex+smoking_0+PC1+PC2+PC3+PCD4+PC5"))  
  model <- lm(formula, data = data_clean2)                          
  summary_model <- summary(model)
  
  mca_row <- coef(summary_model)["mca_status", ]
  data.frame(
    Protein = protein,
    Beta = mca_row["Estimate"],
    SE = mca_row["Std. Error"],
    P_Value = mca_row["Pr(>|t|)"]
  )
})


result_gp_mca <- do.call(rbind, result_list)

# Meta-JAK2&CVD

In [ ]:
install.packages("metafor")
library(metafor)

In [ ]:
meta_data <- data.frame(
  study = c("U", "A", "B"),
  beta = c(-1.5333, -0.03018827, -1.835992),
  se = c(0.4868, 0.2190812, 0.7468469)
)

In [ ]:
exp(-1.835992)

In [ ]:
meta_analysis <- rma(
  yi = beta,       
  sei = se,        
  data = meta_data,
  method = "REML"  
)

summary(meta_analysis)

In [ ]:
forest(meta_analysis)